In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
magic_gamma_telescope = fetch_ucirepo(id=159) 
  
# data (as pandas dataframes) 
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 
  
# metadata 
print(magic_gamma_telescope.metadata) 
  
# variable information 
print(magic_gamma_telescope.variables) 

{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging technique. Cherenkov gamm

In [2]:
import tensorflow as tf
from tensorflow import keras
import sklearn
from keras import layers


In [3]:
# Capa personalizada que implementa una combinación de polinomios de Legendre
class PolynomialDense(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernels = []
        for d in range(1, self.degree + 1):
            w = self.add_weight(
                shape=(input_dim, self.units),
                initializer='glorot_uniform',
                trainable=True,
                name=f'kernel_degree_{d}'
            )
            self.kernels.append(w)

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
                name='bias'
            )

    def call(self, inputs):

        #Asumimos que los inputs ya están normalizados en [-1, 1]
        x = inputs

        # --- 2. Generamos la base de Legendre
        # P0(x)
        P0 = tf.ones_like(x)

        if self.degree == 0:
            features = [P0]
        else:
            # P1(x)
            P1 = x
            features = [P1]

            if self.degree > 1:
                Pnm2 = P0
                Pnm1 = P1

                for n in range(2, self.degree + 1):
                    Pn = ((2.0 * n - 1.0) * x * Pnm1 - (n - 1.0) * Pnm2) / n
                    features.append(Pn)

                    Pnm2 = Pnm1
                    Pnm1 = Pn

        # --- 3. Combinación lineal con los kernels
        output = 0.0

        for phi, W in zip(features, self.kernels):
            output += tf.matmul(phi, W)

        if self.use_bias:
            output += self.bias

        return output


In [4]:
class PolynomialDense2(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense2, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]
        # El kernel debe cubrir (grado + 1) si incluyes P0, o (grado) si empiezas en P1
        # Aquí usaremos desde P1 hasta P_degree
        self.kernel = self.add_weight(
            shape=(input_dim * self.degree, self.units),
            initializer=tf.keras.initializers.GlorotUniform(),
            trainable=True,
            name="kernel"
        )

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer="zeros",
                trainable=True,
                name="bias"
            )

    def call(self, inputs):
        # Aseguramos que los inputs sean float32
        x = tf.cast(inputs, self.compute_dtype)
        
        # P0 = 1, P1 = x
        p_nm2 = tf.ones_like(x)
        p_nm1 = x
        
        features = [p_nm1] # Empezamos con grado 1

        for n in range(2, self.degree + 1):
            n_float = tf.cast(n, self.compute_dtype)
            # Fórmula de recurrencia de Legendre
            p_n = ((2.0 * n_float - 1.0) * x * p_nm1 - (n_float - 1.0) * p_nm2) / n_float
            features.append(p_n)
            
            p_nm2 = p_nm1
            p_nm1 = p_n

        # Concatenamos: (batch, input_dim * degree)
        # Esto genera: [x_poly1, x_poly2, ..., x_poly_degree]
        phi = tf.concat(features, axis=-1)

        output = tf.matmul(phi, self.kernel)

        if self.use_bias:
            output = tf.nn.bias_add(output, self.bias)

        return output

In [5]:
#Dividimos el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.3, random_state=1)

In [6]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((13314, 10), (5706, 10), (13314, 1), (5706, 1))

In [7]:
#Normalizamos los datos
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
#Cambiamos las etiquetas a 0 y 1
y_train = (y_train == 'g').astype(int)
y_test = (y_test == 'g').astype(int)

In [9]:
input_dim = X_train.shape[1]

#Modelo lineal
inputLinear = keras.Input(shape=(input_dim,))
x1 = layers.Dense(32, activation='relu')(inputLinear)
x2 = layers.Dense(16, activation='relu')(x1)
outputLinear = layers.Dense(3, activation='softmax')(x2)

#MODELO DE RED POLINÓMICA DE LEGENDRE

#Modelo polinómico grado 2

inputPoli = keras.Input(shape=(input_dim,))

x = PolynomialDense2(32, degree=2)(inputPoli)
x = layers.Activation('tanh')(x)
x = layers.Dense(16, activation='tanh')(x)

outputPoli = layers.Dense(3, activation='softmax')(x)

#Modelo polinómico grado 3
inputPoli3 = keras.Input(shape=(input_dim,))
x3 = PolynomialDense2(32, degree=3)(inputPoli3)
x3 = layers.Activation('tanh')(x3)
x3 = layers.Dense(16, activation='tanh')(x3)
outputPoli3 = layers.Dense(3, activation='softmax')(x3)

#Modelo polinómico grado 4
inputPoli4 = keras.Input(shape=(input_dim,))
x4 = PolynomialDense2(32, degree=4)(inputPoli4)
x4 = layers.Activation('tanh')(x4)
x4 = layers.Dense(16, activation='tanh')(x4)
outputPoli4 = layers.Dense(3, activation='softmax')(x4)

In [10]:
#Guardamos ambos modelos
modelLinear  = keras.Model(inputs=inputLinear, outputs=outputLinear)
modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli) 
modelPoli3 = keras.Model(inputs=inputPoli3, outputs=outputPoli3)
modelPoli4 = keras.Model(inputs=inputPoli4, outputs=outputPoli4)



In [11]:
modelPoli3.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ polynomial_dense2_1             │ (None, 32)             │           992 │
│ (PolynomialDense2)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,571 (6.14 KB)

 Trainable params: 1,571 (6.14 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
#Compilamos ambos modelos
modelLinear.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli4.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [13]:
epochs = 150

In [14]:
#Añadimos Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
#Entrenamos el modelo
historyLinear = modelLinear.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli = modelPoli.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli3 = modelPoli3.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli4 = modelPoli4.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])

Epoch 1/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.7858 - loss: 0.4738 - val_accuracy: 0.7932 - val_loss: 0.4288
Epoch 2/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8243 - loss: 0.3972 - val_accuracy: 0.8293 - val_loss: 0.3836
Epoch 3/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8377 - loss: 0.3772 - val_accuracy: 0.8395 - val_loss: 0.3705
Epoch 4/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8466 - loss: 0.3646 - val_accuracy: 0.8370 - val_loss: 0.3729
Epoch 5/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8529 - loss: 0.3578 - val_accuracy: 0.8488 - val_loss: 0.3551
Epoch 6/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8543 - loss: 0.3534 - val_accuracy: 0.8498 - val_loss: 0.3594
Epoch 7/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8556 - loss: 0.3490 - val_accuracy: 0.8489 - val_loss: 0.3545
Epoch 8/150
833/833 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8598 - loss: 0.3444 - val_acc

In [16]:
#Comparamos los resultados de ambos modelos
lossLinear, accLinear = modelLinear.evaluate(X_test, y_test, verbose=0)
lossPoli, accPoli = modelPoli.evaluate(X_test, y_test, verbose=0)
lossPoli3, accPoli3 = modelPoli3.evaluate(X_test, y_test, verbose=0)
lossPoli4, accPoli4 = modelPoli4.evaluate(X_test, y_test, verbose=0)

print(f'Modelo Lineal - Loss: {lossLinear:.4f}, Accuracy: {accLinear:.4f}')
print(f'Modelo Polinómico Grado 2 - Loss: {lossPoli:.4f}, Accuracy: {accPoli:.4f}')
print(f'Modelo Polinómico Grado 3 - Loss: {lossPoli3:.4f}, Accuracy: {accPoli3:.4f}')
print(f'Modelo Polinómico Grado 4 - Loss: {lossPoli4:.4f}, Accuracy: {accPoli4:.4f}')

Modelo Lineal - Loss: 0.3217, Accuracy: 0.8628
Modelo Polinómico Grado 2 - Loss: 0.4004, Accuracy: 0.8368
Modelo Polinómico Grado 3 - Loss: 0.3837, Accuracy: 0.8431
Modelo Polinómico Grado 4 - Loss: 0.3761, Accuracy: 0.8444
